In [3]:
# Conversation Last Speaker Count

import json

# ==========================
# Input File
# ==========================
INPUT_FILE = r"./V6A/V6A.jsonl" 

user_count = 0
other_count = 0
error_count = 0

with open(INPUT_FILE, "r", encoding="utf-8") as f:

    for line_num, line in enumerate(f, start=1):

        line = line.strip()

        if not line:
            continue

        try:
            sample = json.loads(line)

            # Handle double-encoded JSON
            if isinstance(sample, str):
                sample = json.loads(sample)

            conversation = sample["input"]["conversation"]

            lines = [x.strip() for x in conversation.split("\n") if x.strip()]

            if not lines:
                continue

            last_line = lines[-1]

            if last_line.startswith("User:"):
                user_count += 1
            else:
                other_count += 1

        except Exception:
            error_count += 1

print("=" * 50)
print("Conversation Last Speaker Count")
print("=" * 50)
print(f"Last message = User : {user_count}")
print(f"Last message = Other: {other_count}")
print(f"Errors              : {error_count}")
print("=" * 50)

Conversation Last Speaker Count
Last message = User : 61104
Last message = Other: 11302
Errors              : 0


In [2]:
import json
import re

# ============================================================
# V5C TURN DISTRIBUTION CHECK
# ============================================================

filename = "./V5C/V5C.jsonl"

turn_4_8 = 0
turn_9_15 = 0
turn_16_24 = 0
turn_25_30 = 0

invalid_turns = 0
errors = 0
total_samples = 0

speaker_pattern = re.compile(
    r"^[A-Za-z][A-Za-z0-9 _-]*:\s*",
    re.MULTILINE
)

with open(filename, "r", encoding="utf-8") as f:

    for line_number, line in enumerate(f, 1):

        if not line.strip():
            continue

        try:
            data = json.loads(line)

            conversation = data["input"]["conversation"]

            if isinstance(conversation, str):

                turns = len(
                    speaker_pattern.findall(conversation)
                )

            elif isinstance(conversation, list):

                turns = len(conversation)

            else:
                raise ValueError("Invalid conversation format")

            total_samples += 1

            # ------------------------------------------------
            # ONLY THE FOUR ALLOWED RANGES
            # ------------------------------------------------

            if 4 <= turns <= 8:
                turn_4_8 += 1

            elif 9 <= turns <= 15:
                turn_9_15 += 1

            elif 16 <= turns <= 24:
                turn_16_24 += 1

            elif 25 <= turns <= 30:
                turn_25_30 += 1

            else:
                invalid_turns += 1

                if invalid_turns <= 20:
                    print(
                        f"INVALID TURN COUNT | "
                        f"Line {line_number} | "
                        f"{turns} turns"
                    )

        except Exception as e:

            errors += 1

            if errors <= 20:
                print(
                    f"ERROR | Line {line_number} | {e}"
                )


# ============================================================
# RESULTS
# ============================================================

print("=" * 80)
print("V5C CONVERSATION TURN DISTRIBUTION")
print("=" * 80)

print(f"Total Samples : {total_samples}")
print(f"Errors        : {errors}")
print()

# ============================================================
# DISTRIBUTION
# ============================================================

print("=" * 80)
print("CONVERSATION LENGTH DISTRIBUTION")
print("=" * 80)

ranges = [
    ("4-8 turns", turn_4_8, 25.0),
    ("9-15 turns", turn_9_15, 45.0),
    ("16-24 turns", turn_16_24, 25.0),
    ("25-30 turns", turn_25_30, 5.0),
]

for name, count, target in ranges:

    percentage = (
        count / total_samples * 100
        if total_samples
        else 0
    )

    print(
        f"{name:<12}: "
        f"{count:>8} "
        f"({percentage:>6.2f}%) "
        f"Target: {target:.2f}%"
    )


# ============================================================
# INVALID
# ============================================================

print()
print("=" * 80)
print("TURN VALIDATION")
print("=" * 80)

print(f"Invalid turn counts : {invalid_turns}")

if invalid_turns == 0:
    print("STATUS: ✅ ALL SAMPLES ARE BETWEEN 4 AND 30 TURNS")
else:
    print("STATUS: ❌ INVALID TURN COUNTS FOUND")


# ============================================================
# TARGET CHECK
# ============================================================

print()
print("=" * 80)
print("TARGET CHECK")
print("=" * 80)

target_counts = {
    "4-8 turns": total_samples * 0.25,
    "9-15 turns": total_samples * 0.45,
    "16-24 turns": total_samples * 0.25,
    "25-30 turns": total_samples * 0.05,
}

actual_counts = {
    "4-8 turns": turn_4_8,
    "9-15 turns": turn_9_15,
    "16-24 turns": turn_16_24,
    "25-30 turns": turn_25_30,
}

for name in target_counts:

    target = target_counts[name]
    actual = actual_counts[name]

    print(
        f"{name:<12}: "
        f"Actual={actual:>8} | "
        f"Target≈{target:>8.0f}"
    )


print("=" * 80)

INVALID TURN COUNT | Line 2150 | 1 turns
INVALID TURN COUNT | Line 2151 | 1 turns
INVALID TURN COUNT | Line 2152 | 1 turns
INVALID TURN COUNT | Line 2153 | 1 turns
INVALID TURN COUNT | Line 2154 | 1 turns
INVALID TURN COUNT | Line 4038 | 1 turns
INVALID TURN COUNT | Line 4039 | 1 turns
INVALID TURN COUNT | Line 4040 | 1 turns
INVALID TURN COUNT | Line 4041 | 1 turns
INVALID TURN COUNT | Line 4042 | 1 turns
INVALID TURN COUNT | Line 4665 | 1 turns
INVALID TURN COUNT | Line 4666 | 1 turns
INVALID TURN COUNT | Line 4667 | 1 turns
INVALID TURN COUNT | Line 4668 | 1 turns
INVALID TURN COUNT | Line 4669 | 1 turns
INVALID TURN COUNT | Line 14555 | 1 turns
INVALID TURN COUNT | Line 14556 | 1 turns
INVALID TURN COUNT | Line 14557 | 1 turns
INVALID TURN COUNT | Line 14558 | 1 turns
INVALID TURN COUNT | Line 14559 | 1 turns
V5C CONVERSATION TURN DISTRIBUTION
Total Samples : 104464
Errors        : 0

CONVERSATION LENGTH DISTRIBUTION
4-8 turns   :    15743 ( 15.07%) Target: 25.00%
9-15 turns  :    

In [5]:
import json
import re

filename = "./V5C/V5C.jsonl"

speaker_pattern = re.compile(
    r"^[A-Za-z][A-Za-z0-9 _-]*:\s*",
    re.MULTILINE
)

shown = 0

with open(filename, "r", encoding="utf-8") as f:

    for line_number, line in enumerate(f, 1):

        if not line.strip():
            continue

        data = json.loads(line)

        conversation = data["input"]["conversation"]

        # Count using current method
        if isinstance(conversation, str):
            turns = len(speaker_pattern.findall(conversation))
        elif isinstance(conversation, list):
            turns = len(conversation)
        else:
            turns = -1

        if turns == 1:

            print("=" * 100)
            print(f"LINE: {line_number}")
            print(f"TYPE: {type(conversation).__name__}")
            print(f"TURN COUNT: {turns}")
            print("-" * 100)
            print(repr(conversation))
            print("-" * 100)
            print("NORMAL TEXT:")
            print(conversation)
            print("=" * 100)

            shown += 1

            if shown >= 10:
                break

LINE: 11
TYPE: str
TURN COUNT: 1
----------------------------------------------------------------------------------------------------
'Colleague: Hey, are you ready for the marketing sync at two? User: Almost, just putting the final touches on the slide deck. Colleague: Great, make sure you include the Q3 conversion metrics we talked about yesterday. User: Good catch, I almost forgot to pull those numbers. Let me add them in right now. Colleague: Awesome, see you in the conference room in ten minutes.'
----------------------------------------------------------------------------------------------------
NORMAL TEXT:
Colleague: Hey, are you ready for the marketing sync at two? User: Almost, just putting the final touches on the slide deck. Colleague: Great, make sure you include the Q3 conversion metrics we talked about yesterday. User: Good catch, I almost forgot to pull those numbers. Let me add them in right now. Colleague: Awesome, see you in the conference room in ten minutes.
LINE: 

In [6]:
import json

filename = "./V5C/V5C.jsonl"

with open(filename, "r", encoding="utf-8") as f:

    for i in range(5):

        data = json.loads(next(f))

        print("=" * 80)
        print("SAMPLE", i + 1)
        print("input type:", type(data["input"]).__name__)
        print("conversation type:",
              type(data["input"]["conversation"]).__name__)
        print("conversation:")
        print(repr(data["input"]["conversation"]))

SAMPLE 1
input type: dict
conversation type: str
conversation:
"Colleague: Hey, are you ready for the quarterly budget review meeting at 2 PM?\nUser: Almost, I'm just finalizing the slide deck for the marketing department expenses.\nColleague: Make sure you highlight the software subscription costs, as the director asked about those yesterday.\nUser: Good catch, I added a dedicated slide for software renewals right after the executive summary.\nColleague: Perfect. Do you want to grab a quick coffee before we head into the conference room?\nUser: Sounds great, let's meet by the breakroom in five minutes."
SAMPLE 2
input type: dict
conversation type: str
conversation:
"Doctor: Good morning, what brings you in today?\nUser: I've had this nagging dry cough for the past two weeks, especially at night.\nDoctor: Any fever, shortness of breath, or body aches?\nUser: No fever or aches, just a mild tickle in my throat that triggers the cough.\nDoctor: Let me listen to your lungs. Take a deep bre